<h1>YOLO Gesture jupyter notebook runner (YOLOv5 branch)</h1>

<h2>1. Checking device and cloning repository</h2>

In [ ]:
!nvidia-smi

In [ ]:
%cd /kaggle/working
!rm -rf cw1
!git clone --branch yolov5 https://github.com/TWWT-ai/RGB-D-Gesture-Recognition.git cw1

In [ ]:
import os
DATA = '/kaggle/input/datasets/temuulenwang/dataset-all/dataset_merged_all'
print('exists:', os.path.exists(DATA))
!ls {DATA} | head

<h2>Fitting the anchors (k-means, 9 = 3 per grid)</h2>
Fitted on the TRAIN persons of the exact split used below (same --seed / --val-frac / --test-frac); train.py refuses the file if the split differs.

In [ ]:
%cd /kaggle/working/cw1/src
!python kmeans_anchors.py \
    --data-root /kaggle/input/datasets/temuulenwang/dataset-all/dataset_merged_all/ \
    --k 9 \
    --val-frac 0.1 \
    --test-frac 0 \
    --seed 42 \
    --out-json /kaggle/working/weights/anchors9.json \
    --fig-out /kaggle/working/figures/anchor_kmeans.png

<h2>Training dataset</h2>

In [ ]:
%cd /kaggle/working/cw1/src
!python train.py \
    --data-root /kaggle/input/datasets/temuulenwang/dataset-all/dataset_merged_all/ \
    --epochs 100 \
    --batch-size 16 \
    --num-workers 2 \
    --lambda-cls 5 \
    --val-frac 0.1 \
    --test-frac 0 \
    --anchors /kaggle/working/weights/anchors9.json \
    --out-dir /kaggle/working/weights/ \
    --seed 42

<h2>2. Evaluating dataset</h2>

In [ ]:
%cd /kaggle/working/cw1/src
!python evaluate.py \
    --data-root /kaggle/input/datasets/temuulenwang/dataset-all/dataset_merged_all \
    --weights /kaggle/working/weights/best.pth \
    --val-frac 0.1 --test-frac 0 --seed 42

<h2>3. Visualising the evaluated dataset</h2>

In [ ]:
%cd /kaggle/working/cw1/src
!python visualize.py \
    --data-root /kaggle/input/datasets/temuulenwang/dataset-all/dataset_merged_all \
    --weights /kaggle/working/weights/best.pth \
    --history /kaggle/working/weights/history.json \
    --out-dir /kaggle/working/figures/ --n-samples 10 --val-frac 0.1 --test-frac 0 --seed 42

import glob
from IPython.display import Image, display
for png in sorted(glob.glob('/kaggle/working/figures/*.png')):
    print(png); display(Image(png, width=500))

<h2>4. Saving trained data</h2>

In [ ]:
# Saving weights
%cd /kaggle/working
!zip -j trained_outputs.zip weights/best.pth weights/last.pth weights/history.json weights/anchors9.json

from IPython.display import FileLink
FileLink('trained_outputs.zip')

In [ ]:
# Saving figures
%cd /kaggle/working
!zip -r figures.zip figures/

from IPython.display import FileLink
FileLink('figures.zip')

<h2>5. Testing on external test set</h2>

In [ ]:
%cd /kaggle/working/cw1/src
!python evaluate.py \
    --data-root /kaggle/input/datasets/temuulenwang/test-set \
    --weights /kaggle/working/weights/best.pth \
    --all-data
!python visualize.py \
    --data-root /kaggle/input/datasets/temuulenwang/test-set \
    --weights /kaggle/working/weights/best.pth \
    --history /kaggle/working/weights/history.json \
    --out-dir /kaggle/working/figures_test/ \
    --n-samples 10 \
    --all-data

import glob
from IPython.display import Image, display
for png in sorted(glob.glob('/kaggle/working/figures_test/*.png')):
    print(png); display(Image(png, width=500))

<h3> 5.1 Saving outputs</h3>

In [ ]:
%cd /kaggle/working
!zip -r figures_test.zip figures_test/

from IPython.display import FileLink
FileLink('figures_test.zip')

<h2>6. Loading from trained dataset</h2>
Upload <code>trained_outputs.zip</code> from section 4 as the Kaggle dataset <code>weights-yolov5</code>. The checkpoint must come from the <b>yolov5</b> branch: checkpoints from other branches are refused (different detection head).

In [ ]:
%cd /kaggle/working/cw1/src
!python evaluate.py \
    --data-root /kaggle/input/datasets/temuulenwang/dataset-all/dataset_merged_all \
    --weights /kaggle/input/datasets/temuulenwang/weights-yolov5/best.pth \
    --val-frac 0.1 --test-frac 0 --seed 42
!python visualize.py \
    --data-root /kaggle/input/datasets/temuulenwang/dataset-all/dataset_merged_all \
    --weights /kaggle/input/datasets/temuulenwang/weights-yolov5/best.pth \
    --history /kaggle/input/datasets/temuulenwang/weights-yolov5/history.json \
    --out-dir /kaggle/working/figures_pre_trained/ --n-samples 10 --val-frac 0.1 --test-frac 0 --seed 42

import glob
from IPython.display import Image, display
for png in sorted(glob.glob('/kaggle/working/figures_pre_trained/*.png')):
    print(png); display(Image(png, width=500))

In [ ]:
%cd /kaggle/working/cw1/src
!python evaluate.py \
    --data-root /kaggle/input/datasets/temuulenwang/test-set \
    --weights /kaggle/input/datasets/temuulenwang/weights-yolov5/best.pth \
    --all-data
!python visualize.py \
    --data-root /kaggle/input/datasets/temuulenwang/test-set \
    --weights /kaggle/input/datasets/temuulenwang/weights-yolov5/best.pth \
    --history /kaggle/input/datasets/temuulenwang/weights-yolov5/history.json \
    --out-dir /kaggle/working/figures_pre_trained_test/ \
    --n-samples 10 \
    --all-data

import glob
from IPython.display import Image, display
for png in sorted(glob.glob('/kaggle/working/figures_pre_trained_test/*.png')):
    print(png); display(Image(png, width=500))